# Mushroom — Logistic Regression

Eén modeltype per notebook. Oorspronkelijke Mushroom-modellen: Andrew; de gedeelde
vergelijking en deze reproduceerbare versie: Codex op verzoek van Jorre.
Jorre rondt de definitieve taakverdeling en GenAI-policy later af (stap 5).

De huidige referentie is de correct uitgevoerde vergelijking op dezelfde 4.100
trainingsrijen, 900 ontwikkeltestrecords en vijf folds. Het model wordt hieronder
werkelijk opnieuw gefit met de vastgelegde configuratie. Alle preprocessing wordt
binnen de fold geleerd. De ontwikkeltest is eerder bekeken en is geen nieuwe
onafhankelijke eindtest. Dit notebook wijzigt de website of bronstudie niet.

We laden de vastgelegde parameters en leggen uit waarom dit model is opgenomen.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
summary = json.loads((ROOT/'MushroomFinal/comparison_correct/comparison_summary.json').read_text(encoding='utf-8'))
key = 'lr_final'
display(pd.Series(summary['classifier_parameters'][key], name='Vastgelegde parameters'))


C                           10
class_weight          balanced
dual                     False
fit_intercept             True
intercept_scaling            1
l1_ratio                  None
max_iter                  2000
multi_class               auto
n_jobs                    None
penalty                     l1
random_state                42
solver               liblinear
tol                     0.0001
verbose                      0
warm_start               False
Name: Vastgelegde parameters, dtype: object

## Oorspronkelijk tuningonderzoek

Deze zoekruimte komt uit het oorspronkelijke projectnotebook. De oorspronkelijke
uitvoer blijft daar bewaard. De eerdere proximity-imputatie was vóór CV geleerd;
daarom gebruiken we die scores niet als eindbewijs. De gecorrigeerde vijf-fold-
evaluatie hieronder leert preprocessing opnieuw binnen iedere fold. Dit is een
reproductie van de vastgelegde configuratie, geen nieuwe gridsearch of modelkeuze.

```python
param_grid_lr = [
    {
        "classifier__C": [0.01, 0.1, 1, 10],
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"]
    },
    {
        "classifier__C": [0.01, 0.1, 1, 10],
        "classifier__solver": ["liblinear"],
        "classifier__penalty": ["l1", "l2"]
    }
]
```

Een lineair model is een nuttige eenvoudiger vergelijking. Schalen, L1-regularisatie en class weights zijn onderdeel van de vastgelegde pipeline.

We fitten vijf trainingsfolds en daarna één model op alle trainingsrijen. De code controleert alle 900 labels en kansen tegen de opgeslagen referentie.

In [2]:
from Final.tools.model_workflow import reproduce
result = reproduce('lr_final')
display(pd.DataFrame(result['folds']))
display(pd.Series({k:v for k,v in result.items() if k != 'folds'}))
assert result['labels_match_original'] and result['max_probability_difference'] < 1e-12


lr_final fold 1 /5


lr_final fold 2 /5


lr_final fold 3 /5


lr_final fold 4 /5


lr_final fold 5 /5


,fold,accuracy,balanced_accuracy,macro_f1,precision_p,recall_p,f1_p,roc_auc,average_precision,tn,fp,fn,tp
0,1,0.635366,0.617480,0.616253,0.518405,0.543408,0.530612,0.684035,0.589020,352,157,142,169
1,2,0.645122,0.622212,0.622436,0.532468,0.527331,0.529887,0.656448,0.598277,365,144,147,164
2,3,0.632927,0.607495,0.607943,0.514851,0.503226,0.508972,0.680323,0.629666,363,147,154,156
3,4,0.663415,0.651613,0.648298,0.550000,0.603226,0.575385,0.689247,0.633478,357,153,123,187
4,5,0.646341,0.638520,0.633412,0.528090,0.606452,0.564565,0.682701,0.605831,342,168,122,188


model                         lr_final
cv_macro_f1_mean              0.625668
cv_macro_f1_std               0.015695
cv_recall_p_mean              0.556729
accuracy                      0.683333
balanced_accuracy             0.667313
macro_f1                      0.666005
precision_p                   0.579096
recall_p                      0.601173
f1_p                          0.589928
roc_auc                       0.702343
average_precision             0.658376
tn                                 410
fp                                 149
fn                                 136
tp                                 205
fit_and_cv_seconds            9.534725
training_rows                     4100
test_rows                          900
labels_match_original             True
max_probability_difference         0.0
sklearn                          1.4.2
dtype: object

De uiteindelijke keuze staat in `09_vergelijking.ipynb`. Een hoge score voor de eetbare klasse is geen bevestiging van eetbaarheid; de website gebruikt een afzonderlijke voorzichtige beslisregel.